### **XGBoost**

- 부스팅의 대표적인 모델
- Gradient Boosting 알고리즘을 고도화한 모델
- 빠른 속도, 높은 성능, 과적합 제어 기능을 제공하는 트리 기반 앙상블 모델
- XGBoost는 sklearn에 포함되어 있지 않다. 별도의 설치가 필요하다.
- 분류, 회귀 모델 모두 존재
- 부스팅의 단점인 병렬화의 어려움을 어느 정도 해결했다.
    - feature별 조건식을 생성하는데 cpu를 병렬로 사용

- **parameter**

    - 1. 학습 제어 관련
        - `n_estimators`
            - default : 100
            - 트리가 생성되는 개수
            - 클수록 성능이 올라가지만 계산이 많아지면서 시간 증가
        
        - `learning_rate`
            - default : 0.3
            - 각 단계별 학습률 (기여도)
            - 일반적으로는 0.01 ~ 0.3 
            - 작을수록 데이터의 단순화, 클수록 복잡화가 이루어짐
        
        - `early_stopping_rounds`
            - default : None
            - fit() 함수의 `eval_set` 매개변수에서 지정된 검증용 데이터셋을 이용한 평가 지표가 n라운드 연속으로 개선되지 않는 경우 학습 중단
            - int형 입력값
            - `n_estimators` 매개변수의 값을 크게 잡고 `early_stopping_rounds`로 최적의 라운드를 검색

        - `objective`
            - 손실함수 지정
            - 분류
                - `binary:logistic`
                    - 이진 분류용 인자값 1
                    - 출력 : 확률
                - `binary:logiraw`
                    - 이진 분류용 인자값 2
                    - 출력 : log-odds 값
                - `multi:softmax`
                    - 다중 클래스 분류
                    - 출력 : 각 클래스의 인덱스(정수)
                - `multi:softprob`
                    - 다중 클래스 분류
                    - 출력 : 각 클래스별 확률
            - 회귀
                - `reg:squarederror`
                    - 평균 제곱 오차 (MSE)
                    - 연속형 데이터 예측에서 사용
                - `reg:absoluteerror`
                    - 평균 절대 오차 (MAE)
                    - 이상치 데이터에 대해 유리
                - `reg:squaredlogerror`
                    - 로그 평균 제곱 오차 (MSLE)
                    - 종속 데이터가 양수이고 값의 범위가 매우 큰 경우
        
        - `eval_metric`
            - default : None
            - 검증시 사용할 평가 지표
            - 조기 종료(`early_stopping_rounds`)와 함께 사용
            - 분류 : `logloss`, `auc`, `error`
                - 다중 분류의 경우 m이 붙음 / ex) `mlogloss`
            - 회귀 : `rmse`, `mae` 등등
    
    2. 트리 구조 제어
        - `max_depth`
            - default : 6
            - 트리의 최대 깊이를 지정
            - 일반적으로 3 ~ 10 정도를 사용
        
        - `min_child_weight`
            - default : 1.0
            - leaf가 분할되기 위한 최소한의 가중치
            - 값이 커지면 보수적인 분할 (과적합의 위험성 내려감)
        
        - `gamma`
            - default : 0.0
            - leaf 추가 분할에 필요한 최소 손실 감소
            - 값이 커지면 불필요한 분할 억제
        
        - `max_leaves`
            - default : None
            - leaf node의 개수를 제한
            - 깊이 대신 개수로 복잡도 제어
        
    3. 샘플링 / 다양성 제어
        - `subsample`
            - default : 1.0
            - 각 트리의 학습에 사용할 샘플의 비율
            - 일반적으로는 0.6 ~ 0.9 사용 → 과적합 방지, 샘플의 다양성
        - `colsample_bytree`
            - default : 1.0
            - 각 트리의 학습에 사용할 feature의 비율
            - 일반적으로는 0.5 ~ 0.9 사용
    
    4. 정규화 / 규제 (회귀)
        - `reg_lambda`
            - default : 1.0
            - L2 정규화 (가중치의 크기를 억제)
            - 값이 큰 경우 안정적인 모델. 과적합 방지
        - `reg_alpha`
            - default : 0.0
            - L1 정규화 (가중치 희소화)
            - feature selection 효과
            - 고차원 데이터에서 사용
    
    5. 데이터 불균형 처리 (분류)
        - `scale_pos_weight`
            - default : 1
            - 양성(1) / 음성(0) 클래스의 불균형 보장
            - neg / pos 비율로 설정 (음성 90%, 양성 10% → 9)
    
    6. 실행의 최적화
        - `tree_method`
            - default : `auto`
            - `auto` : 자동선택
            - `hist` : 대규모 데이터의 빠른 학습
            - `gpu_hist` : GPU 가속 → 대규모/고차원 데이터에서 권장
        
        - `max_bin`
            - default = 256
            - 히스토그램의 버킷(구간)의 수를 지정
            - 값이 커지면 정확도가 상승하지만 메모리의 사용량과 시간이 증가
    
    - hyper parameter 튜닝의 일반적인 순서
        - **1순위**. 모델 성능
            - `n_estimator`(트리의 개수), `learning_rate`(기여도)
            - `max_depth` (최대 깊이)
            - `min_child_weight` (최소 가중치의 합)
        
        - **2순위**. 과적합/일반화
            - `gamma` (손실)
            - `subsample` (학습에 사용할 데이터의 비율)
            - `colsample_bytree` (학습에 사용할 feature의 비율)
        
        - **3순위**. 세밀한 조정/특수 케이스
            - `reg_lambda`, `reg_alpha` (정규화)
            - `scale_pos_weight` (데이터 불균형)
            - `tree_method`, `max_bin` (실행 최적화)


- 속성
    - `feature_importances_`
        - 학습된 모델에서 feature들의 중요도
    
    - `best_score`, `best_iteration`, `best_n_tree_rounds`
        - 조기 종료 이용시에 사용 가능한 속성
        - 최적의 결과 값들을 보여줌
    
    - `eval_result`
        - 학습 중 평가 지표 기록
    
    - `classes_`
        - 학습된 class의 목록

    - `n_features_in_`
        - 입력된 feature의 수

- 메서드
    - `fit(X, y, options)`
        - 모델에 학습 (학습 횟수(epoch)마다 검증이 가능)
        - options 매개변수
            - `eval_set`
                - 검증 데이터셋 지정
                - 학습 로그와 조기 종료에서 사용
            - `verbose`
                - default : None
                - `True` : 각 라운드마다의 성능 출력
                - `10`(int) : 10라운드마다 성능을 출력

    - `predict(X)`
        - 모델의 예측
    
    - `score(X, y)`
        - 분류: 정확도 / 회귀: R² Score
    
    - `save_model(filename)` / `load_model(filename)`
        - 모델을 저장하거나 불러오기

### **코드**

In [ ]:
# XGBoost 라이브러리 설치

# !pip install xgboost

In [ ]:
from xgboost import XGBClassifier, XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, \
        mean_squared_error, r2_score
import pandas as pd
import numpy as np

In [6]:
body = pd.read_csv('../data/bodyPerformance.csv')
body.head(3)

,age,gender,height_cm,weight_kg,body fat_%,diastolic,systolic,gripForce,sit and bend forward_cm,sit-ups counts,broad jump_cm,class
0,27.0,M,172.3,75.24,21.3,80.0,130.0,54.9,18.4,60.0,217.0,C
1,25.0,M,165.0,55.80,15.7,77.0,126.0,36.4,16.3,53.0,229.0,A
2,31.0,M,179.6,78.00,20.1,92.0,152.0,44.8,12.0,49.0,181.0,C


In [7]:
# 성별/등급을 수치형 데이터로 변환

body['gender'] = np.where(
    body['gender'] == 'M', 0, 1
)

body['class'] = body['class'].map({'A': 0, 'B': 1, 'C': 2, 'D': 3})

body.head(3)

,age,gender,height_cm,weight_kg,body fat_%,diastolic,systolic,gripForce,sit and bend forward_cm,sit-ups counts,broad jump_cm,class
0,27.0,0,172.3,75.24,21.3,80.0,130.0,54.9,18.4,60.0,217.0,2
1,25.0,0,165.0,55.80,15.7,77.0,126.0,36.4,16.3,53.0,229.0,0
2,31.0,0,179.6,78.00,20.1,92.0,152.0,44.8,12.0,49.0,181.0,2


In [8]:
X = body.drop('class', axis = 1)
y = body['class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 42, stratify = y)

In [9]:
# XGBoost 분류 모델을 생성 (다중 분류 모델 → softmax 기반)

clf = XGBClassifier(
    n_estimators = 1000,
    learning_rate = 0.05,
    objective = 'multi:softprob',
    eval_metric = 'mlogloss',
    max_depth = 5,
    min_child_weight = 2,
    subsample = 0.8,
    colsample_bytree = 0.8,
    early_stopping_rounds = 50,
    random_state = 42,
    tree_method = 'hist'
)

In [11]:
clf.fit(X_train, y_train, eval_set = [(X_test, y_test)], verbose = 50)

[0]	validation_0-mlogloss:1.35522
[50]	validation_0-mlogloss:0.83340
[100]	validation_0-mlogloss:0.72061
[150]	validation_0-mlogloss:0.67326
[200]	validation_0-mlogloss:0.64875
[250]	validation_0-mlogloss:0.63498
[300]	validation_0-mlogloss:0.62563
[350]	validation_0-mlogloss:0.62029
[400]	validation_0-mlogloss:0.61625
[450]	validation_0-mlogloss:0.61405
[500]	validation_0-mlogloss:0.61321
[550]	validation_0-mlogloss:0.61250
[591]	validation_0-mlogloss:0.61228


,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",50
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method="

In [12]:
print("Best Iteration: ", clf.best_iteration)
print("Best Score: ", clf.best_score)

Best Iteration:  541
Best Score:  0.6121380169092916


In [13]:
# 실제 예측
clf_pred = clf.predict(X_test)

In [15]:
print(confusion_matrix(y_test, clf_pred))
print()
print(classification_report(y_test, clf_pred))

[[900  97   5   2]
 [229 620 130  25]
 [ 89 193 686  37]
 [ 13  48 108 836]]

              precision    recall  f1-score   support

           0       0.73      0.90      0.81      1004
           1       0.65      0.62      0.63      1004
           2       0.74      0.68      0.71      1005
           3       0.93      0.83      0.88      1005

    accuracy                           0.76      4018
   macro avg       0.76      0.76      0.76      4018
weighted avg       0.76      0.76      0.76      4018



##### train / validation / test로 분할

In [16]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 42, stratify = y)

In [17]:
X_vali, X_test, y_vali, y_test = train_test_split(X_test, y_test, test_size=0.5, random_state=42, stratify=y_test)

In [18]:
clf = XGBClassifier(
    n_estimators = 1000,
    learning_rate = 0.05,
    objective = 'multi:softprob',
    eval_metric = 'mlogloss',
    max_depth = 5,
    min_child_weight = 2,
    subsample = 0.8,
    colsample_bytree = 0.8,
    early_stopping_rounds = 50,
    random_state = 42,
    tree_method = 'hist'
)

In [19]:
clf.fit(X_train, y_train, eval_set = [(X_vali, y_vali)], verbose = 50)

[0]	validation_0-mlogloss:1.35525
[50]	validation_0-mlogloss:0.84301
[100]	validation_0-mlogloss:0.73383
[150]	validation_0-mlogloss:0.68888
[200]	validation_0-mlogloss:0.66614
[250]	validation_0-mlogloss:0.65273
[300]	validation_0-mlogloss:0.64388
[350]	validation_0-mlogloss:0.63908
[400]	validation_0-mlogloss:0.63424
[450]	validation_0-mlogloss:0.63262
[500]	validation_0-mlogloss:0.63244
[550]	validation_0-mlogloss:0.63176
[582]	validation_0-mlogloss:0.63163


,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",50
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method="

In [20]:
# 실제 예측
clf_pred = clf.predict(X_test)

In [21]:
print(confusion_matrix(y_test, clf_pred))
print()
print(classification_report(y_test, clf_pred))

[[456  43   3   0]
 [115 310  63  14]
 [ 43  91 352  17]
 [  7  22  48 425]]

              precision    recall  f1-score   support

           0       0.73      0.91      0.81       502
           1       0.67      0.62      0.64       502
           2       0.76      0.70      0.73       503
           3       0.93      0.85      0.89       502

    accuracy                           0.77      2009
   macro avg       0.77      0.77      0.77      2009
weighted avg       0.77      0.77      0.77      2009



##### 다시 돌아와서

In [50]:
boston = pd.read_csv('../csv/boston.csv')
boston.info()

<class 'pandas.DataFrame'>
RangeIndex: 506 entries, 0 to 505
Data columns (total 14 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   CRIM     506 non-null    float64
 1   ZN       506 non-null    float64
 2   INDUS    506 non-null    float64
 3   CHAS     506 non-null    float64
 4   NOX      506 non-null    float64
 5   RM       506 non-null    float64
 6   AGE      506 non-null    float64
 7   DIS      506 non-null    float64
 8   RAD      506 non-null    float64
 9   TAX      506 non-null    float64
 10  PTRATIO  506 non-null    float64
 11  B        506 non-null    float64
 12  LSTAT    506 non-null    float64
 13  Price    506 non-null    float64
dtypes: float64(14)
memory usage: 55.5 KB


In [51]:
X = boston.drop('Price', axis = 1)
y = boston['Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [52]:
# XGBoost 회귀 모델 생성

reg = XGBRegressor()
reg.fit(X_train, y_train, eval_set = [(X_test, y_test)], verbose = 50)


[0]	validation_0-rmse:6.73571


[50]	validation_0-rmse:2.62320
[99]	validation_0-rmse:2.62854


,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_met

In [53]:
reg_pred = reg.predict(X_test)

In [54]:
print("MSE: ", mean_squared_error(y_test, reg_pred))
print("R2: ", r2_score(y_test, reg_pred))

MSE:  6.909231565384943
R2:  0.9057837838492537


In [57]:
reg = XGBRegressor(
    random_state = 42,
    n_estimators = 1000,
    learning_rate = 0.01,
    subsample = 1.0,
    colsample_bytree = 1.0,
    max_depth = 10,
    reg_lambda = 1.0,
    early_stopping_rounds = 10,
    objective = 'reg:squarederror',
    min_child_weight = 3
)

In [58]:
reg.fit(X_train, y_train, eval_set = [(X_test, y_test)], verbose = 50)
reg_pred = reg.predict(X_test)
print("MSE: ", mean_squared_error(y_test, reg_pred))
print("R2: ", r2_score(y_test, reg_pred))
print(reg.best_iteration)

[0]	validation_0-rmse:8.59449


[50]	validation_0-rmse:5.99944
[100]	validation_0-rmse:4.50541
[150]	validation_0-rmse:3.57515
[200]	validation_0-rmse:3.11718
[250]	validation_0-rmse:2.99559
[300]	validation_0-rmse:2.88039
[350]	validation_0-rmse:2.82308
[400]	validation_0-rmse:2.80680
[450]	validation_0-rmse:2.79574
[500]	validation_0-rmse:2.78870
[524]	validation_0-rmse:2.78779
MSE:  7.768548736042205
R2:  0.8940658943087862
514
